# 12 — Per-subject stacker predictions (pooled + LOCO)

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from analysis.lib.data import (load_label, restricted_pool,
                               build_demographics_matrix, identify_binary_targets)
from analysis.lib.stacking import process_stacking_fold
from analysis.lib.cv import pooled_cv, loco_cv

DATA_DIR   = PROJECT_ROOT / "data"
LACUNA_DIR = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
POOLED_OUT = PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "pooled" / "per_subject_predictions.csv"
EXP_DIR    = PROJECT_ROOT / "outputs" / "subject_level_analyses"
EXP_DIR.mkdir(parents=True, exist_ok=True)
POOLED_OUT.parent.mkdir(parents=True, exist_ok=True)
ALL7_OUT = EXP_DIR / "pooled_all7_per_subject_predictions.csv"
LOCO_OUT = EXP_DIR / "loco_psci_per_subject_predictions.csv"

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

In [ ]:
clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
acute_pool = restricted_pool(LACUNA_DIR, clinical_df, all_of=["acuteinfarct"])
clinical = clinical_df.loc[acute_pool]
binary_targets = identify_binary_targets(clinical)

lesion_acute = load_label(LACUNA_DIR, "acuteinfarct", "rd",  "damagepct").reindex(acute_pool)
flnm_acute   = load_label(LACUNA_DIR, "acuteinfarct", "fnm", "tmap").reindex(acute_pool)
slnm_acute   = load_label(LACUNA_DIR, "acuteinfarct", "snm", "disconnectionpct").reindex(acute_pool)

subject_ids = list(acute_pool)
demo_for_subgroup = clinical[["demographics-age", "demographics-sex",
                              "demographics-cohort"]].copy()
view_label_order = ["baseline", "lesion_acute", "flnm_acute", "slnm_acute"]


def collect_predictions(X_demographics, splits_fn, targets):
    """Run the 4-view stacker with return_test_scores=True; long per-fold rows."""
    all_rows = []
    for target in targets:
        y_series = clinical[target]
        valid_mask = y_series.notna()
        valid_idx_in_pool = np.where(valid_mask.values)[0]
        y = y_series[valid_mask].values
        X_base = X_demographics[valid_mask.values]
        n_demo = X_base.shape[1]
        sub_ids_valid = [subject_ids[i] for i in valid_idx_in_pool]

        view_configs = [
            ("baseline",     X_base),
            ("lesion_acute", np.hstack([X_base, lesion_acute.loc[valid_mask].values])),
            ("flnm_acute",   np.hstack([X_base, flnm_acute.loc[valid_mask].values])),
            ("slnm_acute",   np.hstack([X_base, slnm_acute.loc[valid_mask].values])),
        ]
        splits = splits_fn(valid_mask, X_base, y)
        print(f"--- {target} (N={len(y)}, {len(splits)} folds) ---")
        fold_results = Parallel(n_jobs=-1)(
            delayed(process_stacking_fold)(i, train, test, view_configs, y, n_demo,
                                           return_test_scores=True)
            for i, (train, test) in enumerate(splits))
        for r in fold_results:
            for local_pos, ti in enumerate(r["test_idx"]):
                row = {"Target": target, "Fold": r["Fold"],
                       "subject_id": sub_ids_valid[ti],
                       "y_true": int(r["y_test"][local_pos]),
                       "meta_proba": float(r["meta_proba_test"][local_pos]),
                       "meta_decision": float(r["meta_decision_test"][local_pos])}
                for label in view_label_order:
                    if label in r["view_labels"]:
                        col_idx = r["view_labels"].index(label)
                        row[f"base_{label}"] = float(r["base_decisions_test"][local_pos, col_idx])
                    else:
                        row[f"base_{label}"] = np.nan
                all_rows.append(row)
    return pd.DataFrame(all_rows)

In [ ]:
# 1. Pooled, all 7 targets
if POOLED_OUT.exists():
    print(f"[skip] {POOLED_OUT} exists")
    preds = pd.read_csv(POOLED_OUT)
else:
    X_demo, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=False)
    cv = pooled_cv(seed=42)
    preds = collect_predictions(
        X_demo, lambda valid_mask, X_base, y: list(cv.split(X_base, y)),
        binary_targets)
    preds = preds.merge(demo_for_subgroup, left_on="subject_id",
                        right_index=True, how="left")
    preds.to_csv(POOLED_OUT, index=False)
    print(f"saved {len(preds)} rows to {POOLED_OUT}")

In [ ]:
# 1b. Aggregate to one row per (target, subject): mean probability across repeats
if ALL7_OUT.exists():
    print(f"[skip] {ALL7_OUT} exists")
else:
    agg = (preds.groupby(["Target", "subject_id"])
           .agg(y_true=("y_true", "first"),
                mean_meta_proba=("meta_proba", "mean"),
                mean_meta_decision=("meta_decision", "mean"),
                n_folds=("Fold", "count"))
           .reset_index()
           .rename(columns={"Target": "target"}))
    agg = agg.merge(demo_for_subgroup, left_on="subject_id",
                    right_index=True, how="left")
    agg = agg.rename(columns={"demographics-cohort": "cohort"})
    agg.to_csv(ALL7_OUT, index=False)
    print(f"saved {len(agg)} rows to {ALL7_OUT}")

In [ ]:
# 2. LOCO, PSCI only (cohort dummies excluded)
if LOCO_OUT.exists():
    print(f"[skip] {LOCO_OUT} exists")
else:
    X_demo_l, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=True)
    loco_splits = list(loco_cv(clinical["demographics-cohort"]))

    def loco_splits_fn(valid_mask, X_base, y):
        valid_idx = np.where(valid_mask.values)[0]
        mapping = {orig: i for i, orig in enumerate(valid_idx)}
        out = []
        for train, test in loco_splits:
            tr = np.array([mapping[o] for o in train if o in mapping], dtype=int)
            te = np.array([mapping[o] for o in test if o in mapping], dtype=int)
            if len(te) > 0 and len(tr) > 0:
                out.append((tr, te))
        return out

    loco_preds = collect_predictions(X_demo_l, loco_splits_fn,
                                     ["cognition-PSCI_present"])
    loco_preds = loco_preds.merge(demo_for_subgroup, left_on="subject_id",
                                  right_index=True, how="left")
    loco_preds = loco_preds.rename(columns={"demographics-cohort": "cohort"})
    loco_preds.to_csv(LOCO_OUT, index=False)
    print(f"saved {len(loco_preds)} rows to {LOCO_OUT}")